# VieNeu-TTS-v2 on vLLM-Omni v0.22.0 + CUDA 13 — Colab T4 runtime test

Serve **`pnnbao-ump/VieNeu-TTS-v2`** (Vietnamese TTS, Qwen3 backbone + NeuCodec
FSQ codec) as a first-class TTS architecture inside **vLLM-Omni v0.22.0 stable**,
running on a **CUDA 13 userspace** on Colab T4 — no FastAPI wrapper, no
monkey-patches, no edits to the checkpoint's `config.json`.

Target command (the one this notebook runs end-to-end):

```bash
vllm serve pnnbao-ump/VieNeu-TTS-v2 --omni --port 8000 --dtype half
```

**Fork + branch:** `justHman/vllm-omni@feat/vieneu-tts-v0.22` — branched from
upstream tag `v0.22.0` (stable, 2026-06-06) with the VieNeu-TTS-v2 integration
re-ported from the older `feat/vieneu-tts` (v0.19.0rc1 / cu128) fork.

**Why v0.22.0 stable + cu13 re-verify:** the cu13 stack (torch 2.11.0+cu130 +
vllm) was proven on Colab T4 with OmniVoice on vllm-omni v0.24.0rc1. This
notebook re-verifies the same cu13 stack against the **v0.22.0 stable** base
(whose precompiled `.so` are from the torch-2.10/cu128-129 era) — the risk
user accepted. If `vllm 0.22.0 + torch 2.11+cu130` import breaks on T4, cell 4
falls back to `vllm==0.24.0` (the proven cu13 stack) and the discrepancy is
noted; the vieneu port code itself is stack-agnostic.

**Pipeline tested here (end-to-end):**
1. Detect Colab runtime + GPU/CUDA (fail fast).
2. Mount Drive → HF cache + UV cache + git tarball cache (skip re-download on rerun).
3. Install cu130 torch stack + vllm (pinned, re-verify gate).
4. Clone the fork, install editable + `[vieneu]` extra (`sea-g2p`, `neucodec`).
5. nvrtc fallback + transformers shim for neucodec on transformers 5.x.
6. Pre-fetch the checkpoint + external NeuCodec (`neuphonic/neucodec`).
7. Inspect `voices.json` preset list.
8. Launch `vllm serve ... --omni` in the background (survives cell interrupt).
9. Wait for `/v1/models`, then POST `/v1/audio/speech` with a preset voice.
10. Play the WAV inline. Verify RIFF/WAVE header + size > 5000 bytes.
11. (Optional) Voice cloning from `ref_audio` + `ref_text`.

> If start-up or inference fails, the failure output is captured in the launch
> cell + `/tmp/vllm_serve.log` — paste it back so the cu13 / serving wiring can
> be fixed.


In [ ]:
# ============================================================
# 0. Preflight — environment detection (fail fast)
# ============================================================
import os, sys, subprocess, time, json, urllib.request

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    pass

print("✓ Colab:", IN_COLAB)
print("✓ Python:", sys.version.split()[0])

# Show CUDA driver vs userspace BEFORE we touch anything.
try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
        text=True).strip()
    print("✓ GPU / driver:", smi)
except Exception as e:
    raise RuntimeError(
        "nvidia-smi failed — this notebook needs a GPU runtime: "
        "Runtime > Change runtime type > T4 GPU. Err: " + str(e))

# Colab CUDA userspace probe: image historically ships libcudart.so.12 only.
def find_cudart():
    hits = []
    import glob
    for root in ["/usr/local/cuda", "/usr/local/cuda*", "/usr/lib",
                 "/usr/lib/x86_64-linux-gnu", "/lib/x86_64-linux-gnu"]:
        for d in glob.glob(root):
            for f in os.listdir(d) if os.path.isdir(d) else []:
                if f.startswith("libcudart.so"):
                    hits.append(os.path.join(d, f))
    return sorted(set(hits))

cudart = find_cudart()
print("✓ libcudart present on image (pre-install):")
for c in cudart[:8]:
    print("   ", c)
if not cudart:
    print("    <none in standard paths — torch 2.11+cu130 ships libcudart.so.13 via the cuda-toolkit pip package>")

PORT = 8000
MODEL = "pnnbao-ump/VieNeu-TTS-v2"
FORK_REPO = "https://github.com/justHman/vllm-omni.git"
FORK_BRANCH = "feat/vieneu-tts-v0.22"
VLLM_OMNI_TAG = "v0.22.0"
# cu13 stack pins — re-verified in this notebook against v0.22.0 base.
VLLM_PIN = "0.22.0"
VLLM_PIN_FALLBACK = "0.24.0"  # proven cu13 stack; used only if 0.22.0+cu130 breaks
print("✓ model:", MODEL)
print("✓ fork:", FORK_REPO, "@", FORK_BRANCH, "(", VLLM_OMNI_TAG, "base )")
print("✓ cu13 stack pin: vllm", VLLM_PIN, "| fallback vllm", VLLM_PIN_FALLBACK)


In [ ]:
# ============================================================
# 1. Mount Drive — but keep HF cache LOCAL for fast checkpoint loading
# ============================================================
# Drive is mounted for general persistence, but we DO NOT point HF_HOME at
# Drive. The stage-1 codec checkpoint is 773 safetensors shards; loading
# them over Drive FUSE does 773 small random reads and on a bad run takes
# 1238s, blowing past vllm-omni's 600s orchestrator startup timeout
# (TimeoutError: Orchestrator did not become ready within 600s). Keeping
# the HF cache on the local Colab SSD makes checkpoint load a fast local
# read. Trade-off: re-download ~600MB each session — a single sequential
# pull is much faster and far more reliable than 773 FUSE random reads.
MOUNTED = False
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        MOUNTED = True
        print("✓ Drive mounted (for general use; HF cache stays local)")
    except Exception as e:
        print("⚠ Drive mount skipped:", e)

# HF cache — keep on local SSD for fast random reads during weight load.
# Default is /root/.cache/huggingface; we just make sure HF_HOME/HF_HUB_CACHE
# point there explicitly so Drive doesn't sneak in via a leftover env.
hf_cache = "/root/.cache/hf_cache"
os.makedirs(hf_cache, exist_ok=True)
os.environ["HF_HOME"] = hf_cache
os.environ["TRANSFORMERS_CACHE"] = os.path.join(hf_cache, "transformers")
os.environ["HF_HUB_CACHE"] = os.path.join(hf_cache, "hub")
print("✓ HF_HOME =", hf_cache, "(local SSD — fast random reads)")


In [ ]:
# ============================================================
# 2. Install uv + cu130 torch stack + vllm  (re-verify cu13 against v0.22.0)
# ============================================================
# COLLAB CUDA 13 STRATEGY (pip-only):
#   - torch 2.11.0+cu130 wheel declares
#     `cuda-toolkit[cublas,cudart,cufft,...,nvrtc,nvtx]==13.0.2` on Linux.
#     That PyPI meta-package pulls cuda-cudart 13.x which SHIPS
#     libcudart.so.13 via pip => no conda, no manual LD_LIBRARY_PATH; the
#     +cu130 wheels are self-contained for the CUDA 13 userspace.
#   - --torch-backend=cu130 makes uv resolve from the cu130 index. NOT `auto`
#     (auto reads driver 580 / "CUDA 13.0" and also lands on cu130, but pin
#     it explicitly to avoid driver-table drift).
#
# RE-VERIFY GATE: v0.22.0's precompiled .so were built for the torch-2.10 /
# cu128-129 era. torch 2.11+cu130 may or may not ABI-match them. This cell
# pins vllm==0.22.0 first; if import below fails, fall back to vllm==0.24.0
# (the proven cu13 stack) and note the discrepancy.
!pip install -q uv
!uv self update 2>/dev/null || true
!pip uninstall -y torch torchvision torchaudio vllm 2>/dev/null || true

import importlib, sys
# Drop any cached torch-family module so the cu130 install is seen fresh.
for _m in [m for m in list(sys.modules) if m.split('.')[0] in ('torch','vllm','torchaudio','torchvision')]:
    del sys.modules[_m]

def install_vllm(pin):
    print(f"\n>>> installing vllm=={pin} on cu130 torch stack")
    !uv pip install --system --torch-backend=cu130 \
        torch==2.11.0 torchaudio==2.11.0 torchvision==0.26.0 vllm=={pin}

try:
    install_vllm(VLLM_PIN)
    import vllm, torch, torchaudio, torchvision
    print("✓ vllm:", vllm.__version__, "| torch:", torch.__version__,
          "| torchaudio:", torchaudio.__version__, "| torchvision:", torchvision.__version__)
    print("✓ torch.version.cuda =", torch.version.cuda,
          "| cuda available:", torch.cuda.is_available())
    assert vllm.__version__.startswith(VLLM_PIN), f"expected vllm {VLLM_PIN}, got {vllm.__version__}"
    assert torch.__version__.startswith("2.11.0"), f"expected torch 2.11.0, got {torch.__version__}"
    assert "+cu130" in torch.__version__, f"expected torch+cu130, got {torch.__version__}"
    assert torch.cuda.is_available(), "CUDA not available — runtime must be T4 GPU"
    print("✓ GPU:", torch.cuda.get_device_name(0))
    ACTIVE_VLLM = VLLM_PIN
except Exception as e:
    print(f"⚠ vllm {VLLM_PIN} + cu130 import FAILED: {e}")
    print(f">>> fallback to vllm {VLLM_PIN_FALLBACK} (proven cu13 stack; branch code is stack-agnostic)")
    for _m in [m for m in list(sys.modules) if m.split('.')[0] in ('torch','vllm','torchaudio','torchvision')]:
        del sys.modules[_m]
    install_vllm(VLLM_PIN_FALLBACK)
    import vllm, torch, torchaudio, torchvision
    print("✓ vllm:", vllm.__version__, "| torch:", torch.__version__)
    assert vllm.__version__.startswith(VLLM_PIN_FALLBACK)
    assert "+cu130" in torch.__version__
    assert torch.cuda.is_available()
    ACTIVE_VLLM = VLLM_PIN_FALLBACK
    print(f"⚠ NOTE: notebook now runs on vllm {VLLM_PIN_FALLBACK}, NOT base {VLLM_PIN}. "
          f"Branch vieneu code is identical; only the notebook stack pin differs.")
print("ACTIVE_VLLM =", ACTIVE_VLLM)


## ⚠ STOP — Runtime > Restart session (one time)

If cell 2's asserts show `torch 2.11.0+cu128` (NOT `+cu130`) and
`torch.version.cuda = 12.x`, the Jupyter kernel cached the **old** torch module
before the cu130 install replaced the files on disk. This is the classic Colab
`sys.modules` trap: the new wheel is installed but invisible to the running
kernel.

**Fix: `Runtime > Restart session`, then re-run cells 1–2.** Do NOT skip this —
a stale torch under a cu130 vllm is the most common silent break here. After
restart, re-run cells 1–2 only (cell 1 re-mounts Drive / re-sets caches); you do
NOT need to re-run the install.


In [ ]:
# ============================================================
# 3. Clone the fork + editable install with [vieneu] extra (fresh clone each run)
# ============================================================
import os, subprocess, sys

CLONE_DIR = "/content/vllm-omni"
# Fresh clone every run — do NOT cache the clone on Drive. A cached tarball
# would freeze buggy old code and block upstream fixes (the load_presets
# voices.json schema fix in particular) from taking effect on rerun. --depth 1
# keeps the clone cheap (~10s).
!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {FORK_BRANCH} {FORK_REPO} {CLONE_DIR}
# Fetch tags so setuptools-scm can resolve the ancestor tag (v0.22.0) and
# report a real version (0.22.0.dev<N>+g<hash>) instead of its 0.1.dev1
# fallback. --depth 1 alone ships NO tags, which is what made vllm_omni
# report "0.1.dev1+g0bdf181ea" in the first runs.
!cd {CLONE_DIR} && git fetch --tags --depth 1 origin {VLLM_OMNI_TAG}
!cd {CLONE_DIR} && git log --oneline -1
!cd {CLONE_DIR} && git describe --tags HEAD || true

# Single editable install with the vieneu extra (adds sea-g2p + neucodec).
# setup.py resolves platform deps dynamically (VLLM_OMNI_TARGET_DEVICE=cuda);
# it does NOT pin vllm, so the vllm from cell 2 stays.
os.environ["VLLM_OMNI_TARGET_DEVICE"] = "cuda"
os.environ["UV_TORCH_BACKEND"] = "cu130"
# Pretend the exact release version so the editable install reports
# "0.22.0" even if the ancestor tag fetch above is flaky on Colab.
os.environ["SETUPTOOLS_SCM_PRETEND_VERSION"] = VLLM_OMNI_TAG.lstrip("v")
!uv pip install --system -e "{CLONE_DIR}[vieneu]" --torch-backend=cu130

# The editable install wrote a .pth into site-packages, but THIS Jupyter kernel
# started before that .pth existed. The `vllm` CLI subprocess (later cell) starts
# fresh and sees it fine; to verify imports HERE without a restart, prepend the
# source tree explicitly.
if CLONE_DIR not in sys.path:
    sys.path.insert(0, CLONE_DIR)
import vllm_omni
print("✓ vllm_omni:", getattr(vllm_omni, "__version__", "<no __version__>"))
import sea_g2p, neucodec
print("✓ sea_g2p + neucodec importable")

# Re-verify torch stack survived the vieneu-extra install (neucodec declares
# torchao>=0.12.0 with no upper bound; assert so a silent bump surfaces here).
import torch
assert torch.__version__.startswith("2.11.0"), f"torch bumped to {torch.__version__} by vieneu extra"
assert "+cu130" in torch.__version__, f"torch lost +cu130: {torch.__version__}"
print("✓ torch still", torch.__version__, "after vieneu extra")


In [ ]:
# ============================================================
# 4. transformers shim (neucodec on transformers 5.x) + nvrtc + warnings filter
# ============================================================
import importlib, sys, ctypes, warnings

# Shim: neucodec 0.0.6 does `from transformers import HubertModel, Wav2Vec2BertModel`.
# On Colab's transformers 5.x the dynamic top-level export scan doesn't surface
# these (the classes live at transformers.models.hubert.modeling_hubert /
# transformers.models.wav2vec2_bert.modeling_wav2vec2_bert). Patch the namespace
# from the submodules so neucodec's top-level import succeeds regardless.
import transformers
for _cls, _sub in [
    ("HubertModel", "transformers.models.hubert.modeling_hubert"),
    ("Wav2Vec2BertModel", "transformers.models.wav2vec2_bert.modeling_wav2vec2_bert"),
]:
    if not hasattr(transformers, _cls):
        try:
            setattr(transformers, _cls, getattr(importlib.import_module(_sub), _cls))
            print(f"  shim: transformers.{_cls} <- {_sub}")
        except Exception as e:
            print(f"  shim FAILED for {_cls}: {e}")

# nvrtc (NVIDIA Runtime Compilation) — neucodec/torchaudio may JIT CUDA kernels
# via libnvrtc at runtime. torch 2.11+cu130 SHOULD bundle it via cuda-toolkit,
# but on this Colab image the bundled .so isn't on the loader path. The official
# nvidia-cuda-nvrtc-cu12 wheel ships libnvrtc.so.12 (cu12 build, loads fine on a
# CUDA-13-capable driver via forward-compat) — install it as the nvrtc source.
# (nvidia-cuda-nvrtc-cu13 on PyPI is a deprecated skeleton, don't use it.)
try:
    ctypes.CDLL("libnvrtc.so.13")
    print("✓ libnvrtc.so.13 loadable (torch-bundled)")
except OSError:
    try:
        ctypes.CDLL("libnvrtc.so.12")
        print("✓ libnvrtc.so.12 already loadable")
    except OSError:
        print("⚠ installing nvidia-cuda-nvrtc-cu12 for libnvrtc")
        !uv pip install --system --torch-backend=cu130 nvidia-cuda-nvrtc-cu12 || pip install nvidia-cuda-nvrtc-cu12
        try:
            ctypes.CDLL("libnvrtc.so.12")
            print("✓ libnvrtc.so.12 loadable after install")
        except OSError as e:
            print(f"⚠ nvrtc not loadable: {e} — codec JIT may fail")

# Suppress upstream deprecation warnings we cannot fix in this branch (they
# come from external pip packages: diffusers Flax, pydub regex, neucodec
# weight_norm, huggingface_hub local_dir_use_symlinks/resume_download). These
# are noise only — they do not affect the serving path.
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*Flax classes are deprecated.*")
warnings.filterwarnings("ignore", message=".*invalid escape sequence.*")
warnings.filterwarnings("ignore", message=".*weight_norm.*deprecated.*")
warnings.filterwarnings("ignore", message=".*local_dir_use_symlinks.*deprecated.*")
warnings.filterwarnings("ignore", message=".*resume_download.*deprecated.*")
import logging
logging.getLogger("diffusers").setLevel(logging.ERROR)

# Sanity import sweep.
for mod in ["sea_g2p", "torchaudio", "torchvision", "neucodec", "vllm_omni"]:
    try:
        importlib.import_module(mod)
        print(f"✓ {mod} importable")
    except Exception as e:
        print(f"✗ {mod} import FAILED:", e)
        raise


In [ ]:
# ============================================================
# 5. Pre-download the checkpoint + external NeuCodec (avoids first-serve timeout)
# ============================================================
from huggingface_hub import snapshot_download

print("Downloading", MODEL, "...")
mp = snapshot_download(
    MODEL,
    allow_patterns=["*.json", "*.txt", "*.safetensors", "*.model",
                    "voices.json", "tokenizer*"],
)
print("✓ checkpoint at:", mp)

print("Downloading neuphonic/neucodec (external NeuCodec) ...")
npp = snapshot_download("neuphonic/neucodec")
print("✓ neucodec at:", npp)


In [ ]:
# ============================================================
# 6. Inspect voices.json presets + probe vllm serve flags
# ============================================================
import json, os

voices_path = os.path.join(mp, "voices.json")
if os.path.exists(voices_path):
    with open(voices_path) as f:
        voices = json.load(f)
    # voices.json is either {preset_name: {...}} or {"voices": {...}, ...meta}
    presets = voices if not isinstance(voices.get("voices"), dict) else voices["voices"]
    print("✓ preset voices:", sorted(presets.keys()) if isinstance(presets, dict) else list(presets))
else:
    print("⚠ voices.json not found at", voices_path)

# Probe whether the vllm serve supports --stage-config-path (legacy path) for
# diagnose; v0.22.0 uses pipeline.yaml auto-discover for vieneu so we usually
# do NOT need this flag.
print("\n>>> vllm serve --help (grep stage) ...")
!vllm serve --help 2>&1 | grep -i stage || true


In [ ]:
# ============================================================
# 7. Launch `vllm serve ... --omni` in the background (survives cell interrupt)
# ============================================================
import subprocess, os, time

LOG = "/tmp/vllm_serve.log"
# Kill any stale server on PORT.
!fuser -k {PORT}/tcp 2>/dev/null || true

# T4 (compute capability 7.5) cannot run bfloat16 (needs >= 8.0). vieneu's
# pipeline.yaml does not set a dtype, so pass --dtype half on the CLI.
# VLLM_OMNI_TARGET_DEVICE=cuda is read by vllm-omni (platform auto-detect bypass);
# vllm itself logs it as "Unknown env var" but that warning is harmless.
env = dict(os.environ)
env["VLLM_OMNI_TARGET_DEVICE"] = "cuda"

cmd = [
    "vllm", "serve", MODEL,
    "--omni",
    "--port", str(PORT),
    "--host", "0.0.0.0",
    "--trust-remote-code",
    "--dtype", "half",
    # codec load on a slow Colab network can exceed the 600s default; raise
    # both per-stage (300s) and total (600s) to 1800s so a slow first-pull
    # doesn't time out the orchestrator.
    "--stage-init-timeout", "1800",
    "--init-timeout", "1800",
    # Skip torch.compile + CUDA-graph capture (saves ~35s warmup on T4 first
    # launch). Trade-off: ~20-30% slower inference. Drop this line for max
    # throughput if you don't mind the longer cold start.
    "--enforce-eager",
]
print("Launching:", " ".join(cmd))
print("Logs ->", LOG)

# start_new_session=True puts the server in its own process group, so a Jupyter
# cell interrupt (Stop button) does NOT kill the server — you can re-tail the
# log and keep talking to the endpoint.
logf = open(LOG, "w", buffering=1)
proc = subprocess.Popen(
    cmd, stdout=logf, stderr=subprocess.STDOUT,
    cwd="/content/vllm-omni", env=env, start_new_session=True,
)
print("✓ server PID:", proc.pid, "(new session group)")


In [ ]:
# ============================================================
# 8. (Interruptible) tail the server log — Stop cell to stop watching, server keeps running
# ============================================================
# `tail -f` runs until you press the cell's Stop button. The server itself
# survives because it was launched in its own session group (cell 7).
!tail -n 200 -f {LOG}


In [ ]:
# ============================================================
# 9. Wait for /v1/models to come up (print log tail if it times out)
# ============================================================
import urllib.request, json, time

def tail(path, n=80):
    try:
        with open(path, errors="replace") as f:
            return "".join(f.readlines()[-n:])
    except Exception as e:
        return f"<could not read {path}: {e}>"

def get_models(timeout=1500):
    url = f"http://localhost:{PORT}/v1/models"
    start = time.time()
    last_err = None
    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    return json.loads(r.read())
        except Exception as e:
            last_err = e
        if proc.poll() is not None:
            raise RuntimeError(
                f"server process exited early (code {proc.returncode}). See {LOG}.\n"
                + tail(LOG))
        time.sleep(5)
    raise RuntimeError(
        f"timed out waiting for {url}. last_err={last_err}. Tail of {LOG}:\n"
        + tail(LOG))

print("Waiting for server (up to 25 min for first-time compile) ...")
models = get_models(timeout=1500)
print("✓ /v1/models:")
print(json.dumps(models, indent=2))


In [ ]:
# ============================================================
# 10. POST /v1/audio/speech (preset voice) and save a WAV
# ============================================================
import requests

TEXT = "Xin chào, đây là giọng nói tiếng Việt từ VieNeu-TTS trên cu13."

def speech(text=TEXT, voice="Ly", response_format="wav"):
    payload = {"model": MODEL, "input": text,
               "voice": voice, "response_format": response_format}
    r = requests.post(f"http://localhost:{PORT}/v1/audio/speech",
                     json=payload, timeout=180)
    if r.status_code != 200:
        raise RuntimeError(f"speech failed: {r.status_code} {r.text[:2000]}")
    return r

r = speech(voice="Ly")
out_wav = "/content/vieneu_vi_demo.wav"
with open(out_wav, "wb") as f:
    f.write(r.content)
print("✓ wrote", out_wav, len(r.content), "bytes")


In [ ]:
# ============================================================
# 11. Verify WAV + play inline
# ============================================================
from IPython.display import Audio, display

with open(out_wav, "rb") as f:
    head = f.read(12)
size = os.path.getsize(out_wav)
print("header bytes:", head)
assert head[:4] == b"RIFF", f"not a RIFF/WAV file: header={head!r}"
assert head[8:12] == b"WAVE", f"not a WAVE file: header={head!r}"
assert size > 5000, f"WAV suspiciously small: {size} bytes"
print(f"✓ valid WAV, {size} bytes")
display(Audio(out_wav, autoplay=False))


In [ ]:
# ============================================================
# 12. (Optional) Voice cloning from ref_audio + ref_text
# ============================================================
# Upload a 1-30s Vietnamese/English clip + its transcript, then uncomment:
#
# from google.colab import files
# up = files.upload()
# ref_path = list(up.keys())[0]
# import base64, requests
# with open(ref_path, "rb") as f:
#     b64 = base64.b64encode(f.read()).decode()
# r = requests.post(f"http://localhost:{PORT}/v1/audio/speech", json={
#     "model": MODEL,
#     "input": "Hãy nói bằng giọng của tôi.",
#     "ref_audio": f"data:audio/wav;base64,{b64}",
#     "ref_text": "<exact transcript of the clip>",
# }, timeout=180)
# if r.status_code == 200:
#     open("/content/vieneu_clone.wav", "wb").write(r.content)
#     print("✓ wrote /content/vieneu_clone.wav", len(r.content), "bytes")
#     from IPython.display import Audio, display
#     display(Audio("/content/vieneu_clone.wav", autoplay=False))
# else:
#     print("✗ clone failed:", r.status_code, r.text[:1500])
